# Mainstream Training Framework API Comparison

> Section 13 worked out how ZeRO's three stages slice memory redundancy across cards level by level, and noted that DDP / FSDP / DeepSpeed / Accelerate sit at different abstraction levels. But staying at the principle level is not enough: by the time we reach the config file and the training loop line, the API shape each framework exposes to the user is not consistent.
>
> This section organizes the APIs of mainstream training frameworks by abstraction level. Starting from PyTorch-native DDP / FSDP / pipeline.sync, we examine their constructor signatures, key parameters, and minimal usage; then move up to DeepSpeed's config-driven model and Accelerate's unified wrapper; finally we discuss the code structure of Megatron-LM as a self-built layer, and the maturity differences across frameworks in MoE scenarios.


What a training framework's API looks like determines the cognitive load when we write training scripts. Low-level APIs (DDP, FSDP) require explicit handling of process groups, devices, and gradient synchronization; high-level APIs (Accelerate, TRL) hide these inside a single object or a config file, at the cost of reduced control over the underlying layer. The goal of this section is to provide an API map organized by abstraction level, so that when readers pick up a new project they can quickly judge "which layer should this scenario use".

Let's first agree on some terminology. **Parallelism strategy** refers to the algorithmic description of sharding dimensions such as data parallelism, tensor parallelism, pipeline parallelism, and expert parallelism; **framework** refers to the concrete implementation that packages these strategies into APIs, such as PyTorch FSDP, DeepSpeed, and Megatron-LM. The same strategy often has multiple implementations, and the differences between implementations mainly show up in configuration style, communication optimization, and how fast they support new hardware features.

All demos in this section run on a single machine with two processes using the gloo backend, or are marked as pseudocode. The focus is on API shape, not performance.


## 1. API Layer Landscape

APIs related to distributed training roughly fall into four layers. Going upward, abstraction increases and flexibility decreases:

| Layer | Representative API | Form | User responsibility |
|:---|:---|:---|:---|
| L0 primitive | `torch.distributed.init_process_group` | process group, collective op | write all-reduce, broadcast yourself |
| L1 parallel module | `DistributedDataParallel`, `FullyShardedDataParallel`, `pipeline.sync.Pipe` | nn.Module wrapper class | assemble the training loop yourself |
| L2 framework layer | DeepSpeed, Accelerate, TRL | engine or accelerator object | config file + short loop |
| L3 self-built layer | Megatron-LM | custom layers + training script | read source, customize |

L0 is the D topic (distributed primitives); this section starts from L1 and moves upward. L3 is qualitatively different from the first three layers: Megatron-LM is not a library you "import and call", but a training code baseline that needs to be forked or deeply integrated.


## 2. DistributedDataParallel (DDP)

DDP is PyTorch's standard implementation of data parallelism. Its core idea was covered in Section 13: each card holds a full copy of the model, and after the backward pass an all-reduce averages the gradients. Here we look at the API shape.

Constructor signature (simplified):

```python
torch.nn.parallel.DistributedDataParallel(
    module,
    device_ids=None,
    output_device=None,
    dim=0,
    broadcast_buffers=True,
    find_unused_parameters=False,
    gradient_as_bucket_view=False,
    static_graph=False,
)
```

The three most commonly adjusted parameters:


In [ ]:
# === DDP key parameter descriptions (no multi-process launched, only API shape) ===
import inspect
import torch.nn.parallel as par

params = {
    "device_ids": (
        "Specifies the list of GPU IDs used by this process. "
        "For single-GPU training pass [local_rank]; for CPU training pass None."
    ),
    "gradient_as_bucket_view": (
        "Exposes the gradient bucket as a view. "
        "When enabled, in-place optimizers can operate on bucket memory directly, "
        "saving one copy; recommended for FP16 training."
    ),
    "find_unused_parameters": (
        "Traverses the autograd graph to find parameters not participating in backward. "
        "Must be enabled for MoE and conditional computation scenarios; "
        "the cost is one extra graph traversal per forward."
    ),
    "static_graph": (
        "Assumes the communication graph topology of each backward is unchanged. "
        "When enabled, it can schedule overlap between communication and computation; "
        "recommended for models with fixed structure."
    ),
}

for name, desc in params.items():
    print(f"{name}:")
    print(f"  {desc}")
    print()


Below is a minimal DDP demo that runs on a single machine with two processes. It uses `torch.multiprocessing` to launch two processes, with the gloo backend performing the all-reduce.


In [ ]:
# === Minimal DDP demo: gloo backend, single machine two processes ===
# This code is fully runnable; it spawns two worker processes
import os
import torch
import torch.multiprocessing as mp
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP


def worker(rank, world_size):
    """Training loop for a single DDP worker."""
    # Use the gloo backend; no NCCL needed on a single machine
    os.environ["MASTER_ADDR"] = "localhost"
    os.environ["MASTER_PORT"] = "29501"
    dist.init_process_group(
        backend="gloo",
        rank=rank,
        world_size=world_size,
    )

    # A toy model: single linear layer
    model = torch.nn.Linear(4, 2, bias=False)
    # DDP wrapper: under CPU mode device_ids is None
    ddp_model = DDP(model, device_ids=None)

    # Different ranks feed different inputs, simulating data parallelism
    torch.manual_seed(rank)
    x = torch.randn(2, 4)
    target = torch.zeros(2, 2)

    loss = ((ddp_model(x) - target) ** 2).mean()
    loss.backward()

    # DDP all-reduces gradients automatically; grads on both ranks should be identical
    grad_rank0 = ddp_model.module.weight.grad.clone()
    print(f"[rank {rank}] loss={loss.item():.4f}, grad[0,0]={grad_rank0[0,0].item():.4f}")

    dist.barrier()
    if rank == 0:
        print("Key observation: the two ranks see different inputs, but after backward the gradients are identical (all-reduce took effect).")
    dist.destroy_process_group()


if __name__ == "__main__":
    # macOS / Linux single machine: spawn directly
    mp.spawn(worker, args=(2,), nprocs=2, join=True)


The applicability boundary of DDP is very clear: when the model fits entirely on a single card it is the first choice, with minimal communication overhead and the easiest debugging. When the model does not fit — for example full training of a 70B model — the fixed per-card overhead alone exceeds single-card memory, DDP cannot start, and we must switch to FSDP or ZeRO-3.

Another scenario where DDP is not appropriate is MoE. The expert layer only activates part of its parameters during each forward; DDP by default assumes all parameters participate in backward, so `find_unused_parameters=True` must be turned on to synchronize correctly. But this switch brings extra overhead, and industrial MoE training generally goes straight to FSDP + expert parallel.


## 3. FullyShardedDataParallel (FSDP)

FSDP is PyTorch's built-in ZeRO-3 implementation, available since PyTorch 1.11. The key difference from DDP is: in DDP each card holds a full copy of the model, while FSDP shards parameters, gradients, and optimizer state across all cards. When parameters are needed during forward and backward, FSDP automatically all-gathers the full layer and discards it once used.

Constructor signature (simplified):

```python
torch.distributed.fsdp.FullyShardedDataParallel(
    module,
    sharding_strategy=None,
    cpu_offload=None,
    auto_wrap_policy=None,
    backward_prefetch=None,
    mixed_precision=None,
    sync_module_states=False,
    forward_prefetch=False,
    use_orig_params=True,
)
```

The three core configuration options:


In [ ]:
# === FSDP core configuration option descriptions ===
from torch.distributed.fsdp import ShardingStrategy

strategies = {
    ShardingStrategy.FULL_SHARD: (
        "Full ZeRO-3. Parameters/gradients/optimizer all sharded. "
        "Lowest memory, heaviest communication."
    ),
    ShardingStrategy.SHARD_GRAD_OP: (
        "Equivalent to ZeRO-2. Shards gradients and optimizer, parameters stay whole. "
        "Lighter communication than FULL_SHARD, less memory than DDP."
    ),
    ShardingStrategy.NO_SHARD: (
        "Degrades to DDP. No sharding, only gradient all-reduce. "
        "Used for debugging or as a comparison baseline."
    ),
    ShardingStrategy._HYBRID_SHARD_ZERO2: (
        "Hybrid strategy: FULL_SHARD within a machine, SHARD_GRAD_OP across machines. "
        "Suited for multi-node clusters, balances intra-node vs inter-node bandwidth."
    ),
}

print("ShardingStrategy options:")
for s, desc in strategies.items():
    print(f"  {s.name}:")
    print(f"    {desc}")
    print()

print("mixed_precision: specifies the dtype combination for param/grad/reduce.")
print("  Typical config: param=BF16, grad=BF16, reduce=BF16 (recommended on A100/H100).")
print()
print("auto_wrap_policy: automatically decides which submodules to wrap with FSDP.")
print("  Commonly size_based_auto_wrap_policy (by parameter count) or transformer_auto_wrap_policy (by layer type).")



# === FSDP configuration example (structure demo, does not actually launch multi-process) ===
import torch
from torch.distributed.fsdp import (
    FullyShardedDataParallel as FSDP,
    MixedPrecision,
    ShardingStrategy,
)
from torch.distributed.fsdp.wrap import size_based_auto_wrap_policy
import functools

# MixedPrecision config: BF16 recommended on A100/H100
mp_policy = MixedPrecision(
    param_dtype=torch.bfloat16,      # params cast to BF16 during computation
    reduce_dtype=torch.bfloat16,     # gradient all-reduce uses BF16
    buffer_dtype=torch.bfloat16,     # tensors registered via register_buffer are also cast
)

# auto_wrap_policy: submodules with more than 1e6 parameters are wrapped individually
wrap_policy = functools.partial(
    size_based_auto_wrap_policy,
    min_num_params=1_000_000,
)

# Pseudocode: in real training, call init_process_group first
fsdp_init_code = '''
model = MyModel().cuda()
model = FSDP(
    model,
    sharding_strategy=ShardingStrategy.FULL_SHARD,
    mixed_precision=mp_policy,
    auto_wrap_policy=wrap_policy,
    use_orig_params=True,        # keep original parameter names for state_dict
    sync_module_states=True,     # sync rank 0 weights to other ranks
)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
'''

print("FSDP initialization structure:")
print(fsdp_init_code)
print("Key observation: the single line FSDP(model) wires up ZeRO-3,")
print("        and the training loop is almost identical to DDP: loss.backward() + optimizer.step().")



In [ ]:
# === FSDP configuration example (structure demo, does not actually launch multi-process) ===
import torch
from torch.distributed.fsdp import (
    FullyShardedDataParallel as FSDP,
    MixedPrecision,
    ShardingStrategy,
)
from torch.distributed.fsdp.wrap import size_based_auto_wrap_policy
import functools

# MixedPrecision config: BF16 recommended on A100/H100
mp_policy = MixedPrecision(
    param_dtype=torch.bfloat16,      # params cast to BF16 during computation
    reduce_dtype=torch.bfloat16,     # gradient all-reduce uses BF16
    buffer_dtype=torch.bfloat16,     # tensors registered via register_buffer are also cast
)

# auto_wrap_policy: submodules with more than 1e6 parameters are wrapped individually
wrap_policy = functools.partial(
    size_based_auto_wrap_policy,
    min_num_params=1_000_000,
)

# Pseudocode: in real training, call init_process_group first
fsdp_init_code = '''
model = MyModel().cuda()
model = FSDP(
    model,
    sharding_strategy=ShardingStrategy.FULL_SHARD,
    mixed_precision=mp_policy,
    auto_wrap_policy=wrap_policy,
    use_orig_params=True,        # keep original parameter names for state_dict
    sync_module_states=True,     # sync rank 0 weights to other ranks
)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
'''

print("FSDP initialization structure:")
print(fsdp_init_code)
print("Key observation: the single line FSDP(model) wires up ZeRO-3,")
print("        and the training loop is almost identical to DDP: loss.backward() + optimizer.step().")


### FSDP2: a new design with per-parameter sharding

Starting from PyTorch 2.4, FSDP2 was introduced as a redesign of the original FSDP (now called FSDP1). FSDP1 uses `nn.Module` as the sharding unit, flattening all parameters of an entire submodule into a single FlatParameter; FSDP2 switches to per-parameter sharding, where each `nn.Parameter` independently manages its own shard.

This change resolves several long-standing issues. First, FlatParameter breaks the structure of `state_dict`, requiring extra reshape logic when saving checkpoints; FSDP2 preserves each parameter's original shape, so checkpoints align directly with the pretrained weight format. Second, in MoE scenarios expert parameters need to be managed independently; once FlatParameter flattens them, expert parallel becomes hard to do; FSDP2's per-parameter granularity supports this naturally. Third, FSDP1 is hard to compose with DTensor (distributed tensor); FSDP2 is built directly on DTensor, so tensor parallelism and pipeline parallelism can be combined orthogonally.

The API shape has also changed. FSDP2 is no longer a wrapper class but a functional call:

```python
import torch.distributed.tensor.parallel as tp
from torch.distributed.fsdp import fully_shard

model = MyModel()
# Call fully_shard on each submodule that needs sharding
for block in model.blocks:
    fully_shard(block)
fully_shard(model)  # wrap the top level as well
```

As of mid-2025, FSDP2 is in a stabilization phase. PyTorch's mainline recommends FSDP2 for new projects, while existing projects continue with FSDP1.


## 4. pipeline.sync.Pipe

Pipeline parallelism slices the model layer by layer into several stages; each stage is placed on a different device, and data flows through like an assembly line. PyTorch provides the `Pipe` class in `torch.distributed.pipeline.sync` for this purpose.

Constructor signature:

```python
torch.distributed.pipeline.sync.Pipe(
    module,
    balance,
    devices=None,
    chunks=1,
    checkpoint="except_last",
)

# module: nn.Sequential, must be a sequential structure
# balance: list[int], how many layers each stage contains
# devices: list[device], which device each stage sits on
# chunks: number of micro-batches; larger improves pipeline utilization but uses more memory
# checkpoint: 'always' / 'except_last' / 'never', gradient checkpointing strategy
```

`Pipe` has fairly hard constraints: the model must be `nn.Sequential`, and it cannot express cross-layer connections like skip connections or cross attention. Modern LLMs almost all use residual connections, so directly using `Pipe` requires rewriting the residuals into each stage's interior — high engineering cost.


In [ ]:
# === Pipe pseudocode: structure demo (a single machine cannot truly launch a multi-device pipeline) ===
pipe_code = '''
import torch.nn as nn
from torch.distributed.pipeline.sync import Pipe

# Assume 4 Transformer blocks, sliced into 2 stages
model = nn.Sequential(
    Block(0), Block(1),   # stage 0 on GPU 0
    Block(2), Block(3),   # stage 1 on GPU 1
)

pipe_model = Pipe(
    model,
    balance=[2, 2],               # stage 0 has 2 layers, stage 1 has 2 layers
    devices=['cuda:0', 'cuda:1'],
    chunks=8,                     # 8 micro-batches to fill the pipeline
    checkpoint='except_last',     # enable gradient checkpointing on all but the last layer
)

out = pipe_model(input_batch)
'''

print("PyTorch Pipe usage:")
print(pipe_code)
print()
print("Why PyTorch's built-in Pipe is rarely used in production:")
reasons = [
    "1. Must be nn.Sequential; cannot express residual / cross-layer connections",
    "2. Does not support 1F1B (one-forward-one-backward) scheduling; bubble ratio is high",
    "3. Megatron-LM / DeepSpeed PP implementations do interleaved scheduling, which is more efficient",
    "4. Megatron-LM's PP is jointly tuned with TP and EP and is the de facto industry standard",
]
for r in reasons:
    print(f"  {r}")


PyTorch's built-in `Pipe` is mainly used for teaching and small-scale experiments. Real large-scale training almost always switches to Megatron-LM's PipelineParallel or DeepSpeed's PP implementation. The latter two support 1F1B scheduling, interleaved schedules, and composition with tensor parallelism, and are better than `Pipe` in both throughput and memory.


## 5. Megatron-LM Style

Megatron-LM is a training code baseline maintained by NVIDIA; it is not a pip-installable library. Its core value is that tensor parallelism (TP) and pipeline parallelism (PP) are encapsulated inside custom layers, so the model code the user writes looks almost like ordinary PyTorch, but the underlying slicing happens automatically.

The keys to TP are the two classes `ColumnParallelLinear` and `RowParallelLinear`. A standard `nn.Linear(in, out)` holds the full weight on every GPU; `ColumnParallelLinear` splits the weight matrix along the output dimension across N cards, so each card only computes its own slice of the output, and the final result is stitched back together with an all-gather.


In [ ]:
# === Core logic of Megatron-LM ColumnParallelLinear (pseudocode showing the sharding idea) ===
import torch
import torch.nn as nn
import torch.distributed as dist


class ColumnParallelLinear(nn.Module):
    """Linear layer sharded along the output dimension (teaching version, communication optimizations omitted).

    A standard Linear has weight shape [out_features, in_features].
    ColumnParallel splits out_features into N parts; each card holds only [out/N, in].
    """

    def __init__(self, in_features, out_features, world_size):
        super().__init__()
        assert out_features % world_size == 0
        self.out_per_partition = out_features // world_size
        self.in_features = in_features
        # Each card stores only its own slice of the weight
        self.weight = nn.Parameter(
            torch.empty(self.out_per_partition, in_features)
        )
        nn.init.normal_(self.weight)

    def forward(self, x):
        # x: [batch, in_features]
        # Each card computes [batch, out_per_partition]
        out_local = x @ self.weight.t()
        # Real Megatron-LM uses all-gather to stitch the full output back
        # Here we only show the sharding dimension
        return out_local


# Demo of the sharding logic: 8 output dimensions sharded across 4 cards
world_size = 4
layer = ColumnParallelLinear(in_features=6, out_features=8, world_size=world_size)

print(f"Original Linear weight shape: [8, 6], {8*6} parameters total")
print(f"ColumnParallel weight shape (per card): {list(layer.weight.shape)},")
print(f"  each card holds {layer.weight.numel()} parameters = total / {world_size}")
print()
print("Key observation: TP shards the weight across N cards,")
print("        each card's forward only computes [batch, out/N],")
print("        and finally all-gather stitches it back to [batch, out].")


`RowParallelLinear` is the symmetric design: it shards along the input dimension; each card holds part of the input dimension but the full output dimension. During forward each card computes `[batch, out]` (full out) locally, then an all-reduce sums the results. A Transformer's FFN typically combines the two: the first layer is `ColumnParallelLinear` (dimension lift), the second is `RowParallelLinear` (dimension reduction), with no communication needed in between.

### Megatron-LM source reading path

The Megatron-LM codebase is large and beginners can easily get lost. The recommended reading path is:

1. **Entry point**: `pretrain_gpt.py` → read the training loop skeleton, understand the three-part structure of a step (forward / backward / optimizer step)
2. **Model**: `megatron/models/gpt/gpt_model.py` → see how GPTModel assembles embedding + transformer blocks + lm_head
3. **Parallel layers**: `megatron/core/tensor_parallel/layers.py` → the implementation of `ColumnParallelLinear` / `RowParallelLinear`; focus on how `_initialize_affine_weight` shards the weight by rank
4. **Attention**: `megatron/core/transformer/attention.py` → see how TP applies to the QKV projection (ColumnParallel) and attention output (RowParallel)
5. **Pipeline parallelism**: `megatron/core/pipeline_parallel/schedules.py` → see how 1F1B scheduling interleaves forward and backward
6. **MoE**: `megatron/core/transformer/moe/` → how expert parallelism composes with TP, PP, DP

After reading the first four steps you will understand the full picture of TP; step five covers PP scheduling; step six enters MoE.


## 6. DeepSpeed ZeRO

DeepSpeed turns ZeRO's Stage selection, offload strategy, and communication optimizations into JSON config items. Section 13 already showed the Stage 2 config; here we fully expand the field structure of `zero_optimization`.

DeepSpeed's training loop has a three-piece set: `deepspeed.initialize` returns (model_engine, optimizer, dataloader), and all subsequent forward / backward / step calls go through model_engine.


In [ ]:
# === Full structure of the DeepSpeed ZeRO config ===
import json

# ZeroStageEnum: 1=shard optimizer, 2=shard gradients+optimizer, 3=shard parameters+gradients+optimizer
zero_stage_3_config = {
    "train_micro_batch_size_per_gpu": 4,
    "gradient_accumulation_steps": 8,
    "zero_optimization": {
        "stage": 3,
        "overlap_comm": True,                  # overlap communication with computation
        "contiguous_gradients": True,          # gradients in contiguous memory
        "stage3_param_persistence_threshold": 10000,  # params smaller than this are not sharded
        "offload_optimizer": {
            "device": "none"                   # none / cpu / nvme
        },
        "offload_param": {
            "device": "none"                   # only available at Stage 3
        }
    },
    "fp16": {
        "enabled": True,
        "loss_scale": 0,                       # 0 means dynamic loss scaling
        "loss_scale_window": 1000,
        "hysteresis": 2,
        "min_loss_scale": 1
    },
    "activation_checkpointing": {
        "partition_activations": True,         # activations are also sharded
        "cpu_checkpointing": False,
        "number_checkpoints": 16
    }
}

print("DeepSpeed ZeRO Stage 3 config:")
print(json.dumps(zero_stage_3_config, indent=2, ensure_ascii=False))
print()
print("Key observations:")
print("  - stage=3 is equivalent to FSDP FULL_SHARD")
print("  - offload_optimizer.device='cpu' = ZeRO-Offload")
print("  - offload_param.device='nvme' = ZeRO-Infinity")
print("  - overlap_comm=True is almost always recommended; improves throughput")


In [ ]:
# === DeepSpeed training loop three-piece set (pseudocode, needs a real environment to run) ===
deepspeed_loop = '''
import deepspeed

# model / optimizer / dataloader already constructed
model_engine, optimizer, _, _ = deepspeed.initialize(
    model=model,
    optimizer=optimizer,
    model_parameters=model.parameters(),
    training_data=train_dataset,
    config="deepspeed_config.json",
)

for step, batch in enumerate(dataloader):
    # forward: use model_engine instead of model
    loss = model_engine(batch)

    # backward: use model_engine.backward instead of loss.backward
    model_engine.backward(loss)

    # step: DeepSpeed internally handles gradient sync + optimizer.step + zero_grad
    model_engine.step()
'''

print("DeepSpeed training loop:")
print(deepspeed_loop)
print("Key differences:")
print("  - loss.backward() -> model_engine.backward(loss)")
print("  - optimizer.step() + zero_grad() -> model_engine.step()")
print("  - gradient accumulation is controlled by gradient_accumulation_steps in the config; no if in the loop")


## 7. Accelerate

Accelerate is a thin wrapper from HuggingFace that hides the launch differences of DDP / FSDP / DeepSpeed behind a single `Accelerator` object. Its design philosophy is: write the training loop like ordinary PyTorch, and let the `Accelerator` inject the distributed logic.

Core methods of `Accelerator`:


In [ ]:
# === Accelerator core method descriptions ===
methods = {
    "accelerator.prepare(*args)": (
        "Wraps model / optimizer / dataloader / scheduler into their distributed versions. "
        "Under DDP mode the model is wrapped as DDP; under FSDP mode it is wrapped as FSDP."
    ),
    "accelerator.backward(loss)": (
        "Replaces loss.backward(). "
        "Under DeepSpeed mode it calls model_engine.backward; "
        "under FSDP mode it handles gradient synchronization."
    ),
    "accelerator.accumulate(model)": (
        "Context manager that handles gradient accumulation automatically. "
        "Works together with gradient_accumulation_steps in the config."
    ),
    "accelerator.print(*args)": (
        "Prints only on rank 0 to avoid duplicate output across processes. "
        "Equivalent to: if accelerator.is_main_process: print(...)."
    ),
    "accelerator.get_state_dict(model)": (
        "Gets the state_dict under FSDP/DeepSpeed, handling sharded weights. "
        "Must be used when saving checkpoints."
    ),
    "accelerator.wait_for_everyone()": (
        "A barrier synchronization. "
        "Ensures all ranks reach the checkpoint under multi-process."
    ),
}

for name, desc in methods.items():
    print(f"{name}:")
    print(f"  {desc}")
    print()


In [ ]:
# === Migration from a "naive training loop" to an "Accelerate training loop" ===
naive_code = '''
# Naive PyTorch training loop (single card)
model = MyModel().cuda()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
for batch in dataloader:
    loss = model(batch)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
'''

accelerate_code = '''
from accelerate import Accelerator

accelerator = Accelerator(
    gradient_accumulation_steps=8,  # gradient accumulation goes here
    mixed_precision="bf16",        # mixed precision goes here
)

model = MyModel()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

# Key line: prepare hands the objects over to the accelerator
model, optimizer, dataloader = accelerator.prepare(
    model, optimizer, dataloader
)

for batch in dataloader:
    with accelerator.accumulate(model):   # automatic gradient accumulation
        loss = model(batch)
        accelerator.backward(loss)        # replaces loss.backward()
        optimizer.step()
        optimizer.zero_grad()
'''

print("Before vs after migration:")
print("=== Naive loop ===")
print(naive_code)
print("=== Accelerate loop ===")
print(accelerate_code)
print("Key observation: only three changes —")
print("  1. one extra accelerator object")
print("  2. prepare() wraps model/optimizer/dataloader")
print("  3. loss.backward() -> accelerator.backward(loss)")
print("The rest of the training logic is fully preserved. Switching between DDP/FSDP/DeepSpeed only requires changing the launch config.")


Accelerate is not a new algorithm; its value is at the engineering level: the same code, combined with `accelerate launch` and different config files, can switch between DDP, FSDP, and DeepSpeed. This is especially convenient when writing open-source training scripts or running multi-backend comparison experiments.

TRL (Transformer Reinforcement Learning) is a higher-level wrapper built on top of Accelerate, specialized for RLHF / DPO / PPO training. Its `SFTTrainer` / `DPOTrainer` encapsulate data preprocessing, the training loop, and checkpointing, so the user only needs to pass in a model and a dataset. The cost is that customization becomes hard — to change a single training detail (such as a custom loss) you need to understand TRL's internal structure.


## 8. Framework Choice for MoE Scenarios

The parallelism strategies for MoE (Mixture of Experts) training are more complex than for dense models. Besides DP / TP / PP, there is also Expert Parallelism (EP): different experts are placed on different GPUs, and each token is routed via all-to-all to the card that hosts the corresponding expert. This topology demands more from a framework's maturity.

Mainstream frameworks differ markedly in MoE support maturity:

| Framework | MoE support maturity | Typical use |
|:---|:---|:---|
| Megatron-LM | Highest (de facto standard) | Large-scale MoE pretraining (reference implementations for Mixtral, DeepSeek-V3, etc.) |
| DeepSpeed-MoE | High | Provides the combination of ZeRO + EP; PR protocol is easy to migrate |
| PyTorch FSDP2 | Medium (per-parameter benefits EP) | FSDP1 + EP is hard; FSDP2 improves but still lags |
| PyTorch DDP | Low | Requires `find_unused_parameters=True`; high overhead |
| Accelerate | Depends on backend | When the backend is Megatron/DeepSpeed, capabilities match the backend |

Megatron-LM's `ExpertParallel` is the de facto industry standard for MoE training. Under `megatron/core/transformer/moe/` it implements the full set of expert token routing, all-to-all communication, auxiliary loss, and load balancing, and optimizes the joint scheduling with TP / PP / DP. The training configurations of open-source models such as Mixtral, DeepSeek-V3, and Qwen-MoE are all based on Megatron-LM or its variants.


In [ ]:
# === Core fields of the Megatron-LM MoE config (structure demo) ===
megatron_moe_config = {
    "num_experts": 64,                    # total number of global experts
    "expert_model_parallel_size": 8,      # EP parallelism: experts distributed across 8 TP groups
    "moe_router_topk": 2,                 # each token is routed to 2 experts
    "moe_aux_loss_coeff": 0.01,           # auxiliary loss weight, balances load
    "moe_z_loss_coeff": 0.001,            # z-loss prevents router saturation
    "moe_grouped_gemm": True,             # use GroupedGEMM to speed up expert computation
    "moe_token_dropping": False,          # modern MoE usually disables this, replacing drop with token reshuffling
}

print("Megatron-LM MoE core config fields:")
for k, v in megatron_moe_config.items():
    print(f"  {k}: {v}")
print()
print("Key observations:")
print("  - expert_model_parallel_size decides how experts are distributed across GPUs")
print("  - moe_router_topk=2 is the typical config for Mixtral/DeepSeek-MoE")
print("  - moe_aux_loss_coeff controls the strength of load balancing")
print("  - moe_grouped_gemm is performance-critical, avoiding the overhead of per-expert GEMM")


DeepSpeed-MoE is DeepSpeed's extension for MoE, providing the combination of ZeRO + EP. Its advantage is being config-driven and compatible with the DeepSpeed ecosystem; its disadvantage is that performance optimizations (such as GroupedGEMM, shared expert) lag behind Megatron-LM. Small- to medium-scale MoE experiments can use DeepSpeed-MoE to get running quickly; once scale grows, they usually migrate to Megatron-LM.

The combination of FSDP1 + EP is awkward: FlatParameter flattens all expert weights, so they cannot be sharded independently. FSDP2's per-parameter sharding improves on this, and PyTorch 2.5+ `torch.distributed.tensor.parallel` has also added experimental expert parallel support, but as of mid-2025 it is still not as mature as Megatron-LM.

Selection recommendation: for MoE training, prioritize Megatron-LM (or internal frameworks customized from it, such as DeepSeek's framework and Alibaba's PaLM framework). During research and prototyping phases, using DeepSpeed-MoE or FSDP2 + hand-written expert routing is also feasible, but be prepared for performance below that of Megatron-LM.


## 9. Decision Tree from Scenario to Framework

Let's condense the selection experience from the previous sections into a decision tree. Given a training task, judge in the following order:

```text
1. Can the model fit entirely in a single GPU's memory?
   Yes -> DDP (highest throughput, simplest)
   No  -> go to 2

2. Is it an MoE model?
   Yes -> Megatron-LM (de facto standard, includes EP)
          If the scale is small and you want quick validation: DeepSpeed-MoE
   No  -> go to 3

3. Do you need tensor parallelism (TP) or pipeline parallelism (PP)?
   Yes (e.g. 100B+ ultra-large models) -> Megatron-LM (TP+PP+DP+EP 3D parallelism)
   No  -> go to 4

4. Only need data parallelism + ZeRO memory savings?
   Prefer PyTorch-native, want fine control    -> FSDP (or FSDP2)
   Prefer config-driven, need offload          -> DeepSpeed
   Want one codebase across multiple backends   -> Accelerate + FSDP/DeepSpeed
```

Real projects often mix and match: write the main script with Accelerate and pick FSDP or DeepSpeed as the backend; for ultra-large models go straight to Megatron-LM without the Accelerate layer (Megatron-LM has its own complete launcher and parallel scheduling).


## Summary

Confirm you understand the following:

- [ ] Training framework APIs fall into four layers: L0 primitive, L1 parallel module, L2 framework layer, L3 self-built layer (Megatron-LM)
- [ ] The three key parameters of DDP: `device_ids`, `gradient_as_bucket_view`, `find_unused_parameters`
- [ ] The difference between FSDP and DDP: parameters, gradients, and optimizer state are all sharded; during forward and backward they are stitched back via all-gather
- [ ] FSDP2 switches to per-parameter sharding, fixing FSDP1's FlatParameter issues, benefiting MoE and checkpointing
- [ ] PyTorch's built-in `Pipe` is rarely used in production; Megatron-LM's PP is more common (supports 1F1B scheduling)
- [ ] Megatron-LM's `ColumnParallelLinear` / `RowParallelLinear` are the encapsulation core of TP
- [ ] DeepSpeed's training loop three-piece set: `deepspeed.initialize` + `model_engine.backward` + `model_engine.step`
- [ ] Accelerate's value is "one codebase across multiple backends"; underneath it is still DDP/FSDP/DeepSpeed
- [ ] The de facto standard for MoE training is Megatron-LM; DeepSpeed-MoE and FSDP2 are alternatives
- [ ] Selection decision tree: fits on a single card -> DDP; MoE -> Megatron-LM; ultra-large models -> Megatron-LM; otherwise -> FSDP/DeepSpeed/Accelerate


## Homework

> You may ask AI to explain the approach, break down the steps, or check the direction, but it is not recommended to have AI "finish this problem for you".

**Homework 1: Identify the framework for each scenario**

Given the three training scenarios below, write down the most appropriate framework name (choose from DDP / FSDP / DeepSpeed / Megatron-LM / Accelerate).

Hint: refer to the decision tree in Section 9.


In [ ]:
# Homework 1: scenario to framework matching
# Scenario A: train a 7B dense model on 8 A100s; a single card cannot hold the full model but TP is not needed
# Scenario B: train a 100B MoE model on 2048 H100s; needs TP + PP + EP
# Scenario C: fine-tune a 1.5B model on 2 consumer GPUs for experiments; want a single codebase that supports both DDP and single-card

answer_a = None  # TODO: fill in the framework name (string, e.g. "DDP")
answer_b = None
answer_c = None

valid = {"DDP", "FSDP", "DeepSpeed", "Megatron-LM", "Accelerate"}
assert answer_a in valid, f"The answer for scenario A must be one of {valid}"
assert answer_b in valid, f"The answer for scenario B must be one of {valid}"
assert answer_c in valid, f"The answer for scenario C must be one of {valid}"

# Scenario A: 7B does not fit on a single card, dense model does not need TP -> FSDP or DeepSpeed are both correct
assert answer_a in {"FSDP", "DeepSpeed"}, (
    "Scenario A hint: 7B not fitting on a single card needs ZeRO-3 level sharding; dense does not need TP"
)
# Scenario B: ultra-large MoE needs TP+PP+EP -> Megatron-LM is the de facto standard
assert answer_b == "Megatron-LM", (
    "Scenario B hint: 100B MoE + TP+PP+EP is only maturely supported by Megatron-LM"
)
# Scenario C: want one codebase across backends -> Accelerate
assert answer_c == "Accelerate", (
    "Scenario C hint: a single codebase supporting both DDP and single-card is Accelerate's core positioning"
)

print("Homework 1 passed:")
print(f"  Scenario A (7B dense, does not fit on a single card): {answer_a}")
print(f"  Scenario B (100B MoE, ultra-large scale):             {answer_b}")
print(f"  Scenario C (small-scale experiment, multi-backend):   {answer_c}")


**Homework 2: Complete the FSDP MixedPrecision config**

Write an FSDP MixedPrecision config with the following requirements: parameters use BF16, gradient all-reduce uses BF16, and buffer also uses BF16.

Hint: refer to the example in Section 3; the three field names are `param_dtype`, `reduce_dtype`, `buffer_dtype`.


In [ ]:
# Homework 2: FSDP MixedPrecision config
import torch
from torch.distributed.fsdp import MixedPrecision

mp_policy = None  # TODO: construct the MixedPrecision object

assert mp_policy is not None, "Please construct the MixedPrecision object first"
assert mp_policy.param_dtype == torch.bfloat16, "param_dtype should be torch.bfloat16"
assert mp_policy.reduce_dtype == torch.bfloat16, "reduce_dtype should be torch.bfloat16"
assert mp_policy.buffer_dtype == torch.bfloat16, "buffer_dtype should be torch.bfloat16"

print("Homework 2 passed:")
print(f"  param_dtype:  {mp_policy.param_dtype}")
print(f"  reduce_dtype: {mp_policy.reduce_dtype}")
print(f"  buffer_dtype: {mp_policy.buffer_dtype}")
print("This config is the standard BF16 mixed precision setting for FSDP on A100/H100.")


**Homework 3: Identify the parameter DDP needs in MoE scenarios**

An MoE model only activates part of its expert parameters during each forward. When using DDP to train MoE, which parameter must be enabled to synchronize gradients correctly?

Hint: this parameter makes DDP traverse the autograd graph and find parameters that did not participate in backward. Refer to Section 2.


In [ ]:
# Homework 3: the key parameter for DDP in MoE scenarios
answer = None  # TODO: fill in the parameter name (string)

valid_params = {
    "find_unused_parameters",
    "find_unused_parameters=True",
    "find_unused_parameters = True",
}
# Normalize: strip spaces and = True
normalized = answer.replace(" ", "").replace("=True", "").lower() if answer else ""
assert normalized == "find_unused_parameters", (
    "Hint: DDP by default assumes all parameters participate in backward; "
    "in MoE scenarios a parameter that traverses the autograd graph must be enabled"
)

print("Homework 3 passed:")
print(f"  MoE + DDP must enable: find_unused_parameters=True")
print("  Reason: MoE only activates part of the experts each time, and the inactive parameters do not participate in backward;")
print("          DDP needs to know this to do gradient synchronization correctly.")
print("  Cost: one extra autograd graph traversal per forward, so industrial MoE")
print("        usually goes straight to FSDP + expert parallel instead of DDP.")


## References

- [PyTorch DDP documentation](https://pytorch.org/docs/stable/generated/torch.nn.parallel.DistributedDataParallel.html)
- [PyTorch FSDP documentation](https://pytorch.org/docs/stable/fsdp.html)
- [FSDP2 design document (PyTorch RFC)](https://github.com/pytorch/pytorch/issues/114299)
- [PyTorch Pipe documentation](https://pytorch.org/docs/stable/pipeline.html)
- [Megatron-LM repository](https://github.com/NVIDIA/Megatron-LM)
- [DeepSpeed documentation](https://www.deepspeed.ai/)
- [HuggingFace Accelerate documentation](https://huggingface.co/docs/accelerate/)
- [HuggingFace TRL documentation](https://huggingface.co/docs/trl/)
- Rajbhandari et al., [ZeRO: Memory Optimizations Toward Training Trillion Parameter Models](https://arxiv.org/abs/1910.02054), 2020
- Shoeybi et al., [Megatron-LM: Training Multi-Billion Parameter Language Models Using Model Parallelism](https://arxiv.org/abs/1909.08053), 2019
